<H3>PRI 2025/26: first project delivery</H3>

**GROUP X**
- name, number
- name, number
- name, number

<H3>Part 0: data loading</H3>

In [1]:

import os
from pathlib import Path

def load_bbc_dataset(root_dir):
    root = Path(root_dir)
    docs = []
    refs = []
    for category in ["business", "entertainment", "politics", "sport", "tech"]:
        art_dir = root / "News Articles" / category
        sum_dir = root / "Summaries" / category
        for art_file in sorted(art_dir.glob("*.txt")):
            sum_file = sum_dir / art_file.name
            if not sum_file.exists():
                continue
            # more robust decoding
            with art_file.open(encoding="utf-8", errors="replace") as f:
                d = f.read().strip()
            with sum_file.open(encoding="utf-8", errors="replace") as f:
                r = f.read().strip()
            docs.append({"id": art_file.stem, "category": category, "text": d})
            refs.append({"id": art_file.stem, "category": category, "summary": r})
    return docs, refs


import spacy

# e.g. 'en_core_web_sm' or a larger model if allowed
nlp = spacy.load("en_core_web_sm", disable=["ner"])

def preprocess_document(text):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

    tokens = []
    noun_phrases = set()
    for sent in doc.sents:
        for token in sent:
            if token.is_stop or token.is_punct or token.is_space:
                continue
            lemma = token.lemma_.lower()
            if lemma:
                tokens.append(lemma)
        for np in sent.noun_chunks:
            phrase = np.text.lower().strip()
            if phrase:
                noun_phrases.add(phrase)

    return {
        "sentences": sentences,
        "tokens": tokens,
        "noun_phrases": list(noun_phrases),
    }
   
   
## test 
docs, refs = load_bbc_dataset("data/BBC News Summary")
print(len(docs), len(refs))
print(docs[0])
print(refs[0])


sample = docs[0]["text"]
pre = preprocess_document(sample)
print("Sentences:", len(pre["sentences"]))
print("Tokens:", len(pre["tokens"]))
print("Noun phrases:", len(pre["noun_phrases"]))
print("First 3 sentences:")
for s in pre["sentences"][:3]:
    print("-", s)
print("Some noun phrases:", pre["noun_phrases"][:10])




2225 2225
{'id': '001', 'category': 'business', 'text': 'Ad sales boost Time Warner profit\n\nQuarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.\n\nThe firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.\n\nTime Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL\'s underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service f

<H3>Part I: demo of facilities</H3>

A) **Indexing** (preprocessing and indexing options)

In [2]:
import time
import sys
from collections import defaultdict, Counter

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer


def estimate_size_bytes(obj, seen=None):
    """
    Very rough recursive size estimator, just to report
    the in-memory footprint of the index.
    """
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)
    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(estimate_size_bytes(k, seen) + estimate_size_bytes(v, seen)
                    for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(estimate_size_bytes(i, seen) for i in obj)
    return size


def build_inverted_index(preprocessed_docs, meta_docs):
    """
    preprocessed_docs: dict doc_id -> {"tokens": [...], ...}
    meta_docs: dict doc_id -> {"category": ..., ...}
    """
    inverted_index = defaultdict(list)  # term -> list of (doc_id, tf)
    doc_stats = {}

    for doc_id, pdata in preprocessed_docs.items():
        tokens = pdata["tokens"]
        category = meta_docs[doc_id]["category"]

        token_counts = Counter(tokens)
        num_tokens = len(tokens)
        num_terms = len(token_counts)

        # fill postings
        for term, tf in token_counts.items():
            inverted_index[term].append((doc_id, tf))

        # basic stats per document
        doc_stats[doc_id] = {
            "num_tokens": num_tokens,
            "num_terms": num_terms,
            "category": category,
        }

    return inverted_index, doc_stats


def compute_bm25_stats(inverted_index, doc_stats, k1=1.2, b=0.75):
    """
    Pre-compute parts needed for BM25 ranking (at document level).
    """
    N = len(doc_stats)
    doc_lengths = np.array([stats["num_tokens"] for stats in doc_stats.values()], dtype=float)
    avgdl = doc_lengths.mean() if N > 0 else 0.0

    # map doc_id -> integer index (useful later)
    doc_ids = list(doc_stats.keys())
    doc_id_to_idx = {d_id: i for i, d_id in enumerate(doc_ids)}

    # IDF per term
    bm25_idf = {}
    for term, postings in inverted_index.items():
        df_t = len(postings)
        # classic BM25 IDF
        bm25_idf[term] = np.log((N - df_t + 0.5) / (df_t + 0.5) + 1)

    bm25_params = {
        "k1": k1,
        "b": b,
        "N": N,
        "avgdl": avgdl,
        "doc_ids": doc_ids,
        "doc_id_to_idx": doc_id_to_idx,
        "doc_lengths": doc_lengths,
        "idf": bm25_idf,
    }
    return bm25_params


def build_tfidf_matrix(preprocessed_docs):
    """
    Build TF-IDF matrix at document level (lexical vector store).
    """
    # join tokens + noun phrases so they share the same vocabulary
    docs_as_strings = []
    doc_ids = []

    for doc_id, pdata in preprocessed_docs.items():
        tokens = pdata["tokens"]
        noun_phrases = pdata["noun_phrases"]
        # simple way: represent each doc as a space-separated string
        # including both word lemmas and noun phrases
        doc_str = " ".join(tokens + noun_phrases)
        docs_as_strings.append(doc_str)
        doc_ids.append(doc_id)

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(docs_as_strings)

    return {
        "vectorizer": vectorizer,
        "matrix": tfidf_matrix,
        "doc_ids": doc_ids,
    }


def build_dense_embeddings(preprocessed_docs, model_name="BAAI/bge-small-en-v1.5"):
    """
    Build sentence-level or document-level embeddings using a lightweight encoder.
    To keep it simple, we start with *document-level* embeddings by concatenating sentences.
    """
    model = SentenceTransformer(model_name)

    doc_ids = []
    texts_for_embedding = []
    for doc_id, pdata in preprocessed_docs.items():
        # here we use the original sentences; you could also
        # use just the first N sentences or a summary-like subset.
        joined_sentences = " ".join(pdata["sentences"])
        doc_ids.append(doc_id)
        texts_for_embedding.append(joined_sentences)
    embeddings = model.encode(texts_for_embedding, convert_to_numpy=True)
    dense_index = {
        "model_name": model_name,
        "doc_ids": doc_ids,
        "embeddings": embeddings,  # shape: (num_docs, dim)
    }
    return dense_index
def indexing(D, args=None):
    """
    Main indexing function requested in the project statement.
    @input:  D - list of dicts with keys {"id", "category", "text"}
             args - optional dict with configuration options
    @output: I (hybrid index), indexing_time (seconds), index_size_bytes
    """
    if args is None:
        args = {}
    t0 = time.time()
    # 1) Preprocess all documents (re-use your existing preprocess_document)
    preprocessed_docs = {}
    meta_docs = {}
    for doc in D:
        doc_id = doc["id"]
        category = doc["category"]
        text = doc["text"]
        pdata = preprocess_document(text)
        preprocessed_docs[doc_id] = pdata
        meta_docs[doc_id] = {"category": category}
    # 2) Build lexical inverted index + doc stats
    inverted_index, doc_stats = build_inverted_index(preprocessed_docs, meta_docs)
    # 3) Compute BM25 statistics for later use
    bm25_params = compute_bm25_stats(inverted_index, doc_stats,
                                     k1=args.get("bm25_k1", 1.2),
                                     b=args.get("bm25_b", 0.75))
    # 4) Build TF-IDF document vectors
    tfidf_index = build_tfidf_matrix(preprocessed_docs)
    # 5) Build dense embeddings (vector index) – you can make this optional via args
    if args.get("use_dense", True):
        dense_index = build_dense_embeddings(
            preprocessed_docs,
            model_name=args.get("dense_model", "BAAI/bge-small-en-v1.5")
        )
    else:
        dense_index = None
    # 6) Pack everything into a single index object
    I = {
        "preprocessed_docs": preprocessed_docs,  # sentences, tokens, noun_phrases
        "doc_stats": doc_stats,                  # per-document metadata
        "inverted_index": inverted_index,        # term -> list of (doc_id, tf)
        "bm25": bm25_params,                     # precomputed stats for BM25
        "tfidf": tfidf_index,                    # vectorizer + sparse matrix
        "dense": dense_index,                    # encoder name + embeddings
    }
    indexing_time = time.time() - t0
    index_size_bytes = estimate_size_bytes(I)
    return I, indexing_time, index_size_bytes

/Users/christinescheisunnevag/Library/Python/3.11/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
I, build_time, mem_bytes = indexing(docs, args={"use_dense": True})
print("Index built in", build_time, "seconds")
print("Approximate index size:", mem_bytes / (1024**2), "MB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11536.19it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built in 166.29459500312805 seconds
Approximate index size: 20.333231925964355 MB


B) **Extractive summarization**

*B.1 Summarization baseline solution: results for a given document*

In [4]:
#code, statistics and/or charts here


*B.2 IR models (TF-IDF, BM25, lightweight encoder LLMs)*

In [5]:
#code, statistics and/or charts here

*B.4 Maximal Marginal Relevance*

In [6]:
#code, statistics and/or charts here

C) **Abstractive summarization**

*C.1 Prompting decoder LLMs*

In [7]:
#code, statistics and/or charts here

*C.2 Reciprocal rank funsion*

In [8]:
#code, statistics and/or charts here

D) **Keyword extraction**

In [9]:
#code, statistics and/or charts here

E) **Evaluation**

In [10]:
#code, statistics and/or charts here

<H3>Part II: questions materials (optional)</H3>

**(1)** Keyword extraction using classic *vs* generative stances

In [11]:
#code, statistics and/or charts here

**(2)** Summarization performance for the overall and category-conditional corpora.

In [12]:
#code, statistics and/or charts here

**...** (additional questions with empirical results)

<H3>END</H3>